# 9e — Defend K-Means: Interesting Misalignments

**Goal:** Find clusters where content-based grouping most disagrees with ArXiv human categorization.  
Reframe NMI=0.34 from "we didn't fully match" to "here's what the category system missed."

**Output:** `../../arxiv-trends-website/src/data/q5_misalignments.json`

**Key metric:** *migration rate* = fraction of papers in a cluster whose ArXiv primary_category domain ≠ the cluster's dominant domain.

**Prerequisites:** `cluster_labels_500d.pkl`, `arxiv_metadata_features.pkl`

In [ ]:
import pickle, json, os
import numpy as np
import pandas as pd
from collections import Counter

DATA = os.path.join(os.path.dirname(os.getcwd()), 'data', 'processed')
OUT  = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'arxiv-trends-website', 'src', 'data')

print('Loading...')
with open(os.path.join(DATA, 'cluster_labels_500d.pkl'), 'rb') as f:
    labels = pickle.load(f)
with open(os.path.join(DATA, 'arxiv_metadata_features.pkl'), 'rb') as f:
    meta = pickle.load(f)

df = pd.DataFrame({'cluster': labels, 'category': meta['primary_category'], 'year': meta['year']})
print(f'Papers: {len(df):,} | Clusters: {df["cluster"].nunique()}')

In [ ]:
def cat_to_domain(cat):
    if not isinstance(cat, str): return 'other'
    if cat.startswith('cs.'): return 'cs'
    if cat.startswith('math.') or cat == 'math-ph': return 'math'
    if any(cat.startswith(p) for p in ['astro-ph','hep-','gr-qc','nucl-','physics.','quant-ph','cond-mat']): return 'physics'
    if cat.startswith('stat.'): return 'stats'
    if cat.startswith('eess.'): return 'eess'
    return 'other'

df['domain'] = df['category'].apply(cat_to_domain)

results = []
for cid in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == cid]
    domain_counts = sub['domain'].value_counts()
    dominant_domain = domain_counts.index[0]
    total = len(sub)
    migrants = total - domain_counts.iloc[0]
    migration_rate = migrants / total
    surprising = [{'category': c, 'count': int(n)} for c, n in sub[sub['domain'] != dominant_domain]['category'].value_counts().head(3).items()]
    results.append({'clusterId': int(cid), 'dominantDomain': dominant_domain, 'migrationRate': round(migration_rate, 4), 'totalPapers': total, 'migrantPapers': migrants, 'surprisingCategories': surprising})

results_df = pd.DataFrame(results).sort_values('migrationRate', ascending=False)
print(results_df[['clusterId','dominantDomain','migrationRate','totalPapers']].head(15).to_string())

In [ ]:
output = {
    'summary': {
        'totalPapers': int(len(df)),
        'avgMigrationRate': round(results_df['migrationRate'].mean(), 4),
        'medianMigrationRate': round(results_df['migrationRate'].median(), 4),
        'highMigrationClusters': int((results_df['migrationRate'] > 0.3).sum())
    },
    'topMisalignments': results_df.head(10).to_dict('records'),
    'clusterMigrationRates': results
}
out_path = os.path.join(OUT, 'q5_misalignments.json')
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved → {out_path}')

## Next step
Add Q5 sub-tab in `results.jsx` showing top misalignments with narrative.  
Key framing: *these disagreements are discoveries, not failures.*